In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [2]:
# Step 1: User inputs folder location, automatically parse structe

# run main() from scan_folder.py
from nade.io.scan_folder import scan_folder_with_metadata

folder_name = PROJECT_ROOT /"tests/reference_data/raw_audio_songmeter/"

files_metadata = scan_folder_with_metadata(folder_name)
files_metadata

2026-09-23 14:30:37,695 - Scanning folder: c:\Users\ram212\PhD\Code\NADE\tests\reference_data\raw_audio_songmeter
2026-09-23 14:30:37,697 - Found 15 .wav files
2026-09-23 14:30:37,700 - Total audio files found: 15


{'count': 15,
 'folder': WindowsPath('c:/Users/ram212/PhD/Code/NADE/tests/reference_data/raw_audio_songmeter'),
 'site_names': ['site_1', 'site_2', 'site_3'],
 'file_metadata': [{'path': 'c:\\Users\\ram212\\PhD\\Code\\NADE\\tests\\reference_data\\raw_audio_songmeter\\site_1\\SMM09034_20230425_074500.wav',
   'site_name': 'site_1',
   'datetime': datetime.datetime(2023, 4, 25, 7, 45)},
  {'path': 'c:\\Users\\ram212\\PhD\\Code\\NADE\\tests\\reference_data\\raw_audio_songmeter\\site_1\\SMM09034_20230425_080000.wav',
   'site_name': 'site_1',
   'datetime': datetime.datetime(2023, 4, 25, 8, 0)},
  {'path': 'c:\\Users\\ram212\\PhD\\Code\\NADE\\tests\\reference_data\\raw_audio_songmeter\\site_1\\SMM09034_20230425_081500.wav',
   'site_name': 'site_1',
   'datetime': datetime.datetime(2023, 4, 25, 8, 15)},
  {'path': 'c:\\Users\\ram212\\PhD\\Code\\NADE\\tests\\reference_data\\raw_audio_songmeter\\site_1\\SMM09034_20230425_083000.wav',
   'site_name': 'site_1',
   'datetime': datetime.datetime

In [ ]:
# Step 2: Sample background audio segments from the files_metadata

from nade.sampling.background import sample_background_audio

sample_background_audio(files_metadata,
                        model_input_length = 3.0,
                        num_background_samples = 20,
                        sampling_type = "random",
                        output_folder = PROJECT_ROOT / "tests/reference_data/background_audio",
                        seed = None,
                        overwrite = False,)

c:\Users\ram212\PhD\Code\NADE\src\nade\sampling\background.py:98: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration = librosa.get_duration(filename=str(path))


'c:\\Users\\ram212\\PhD\\Code\\NADE\\tests\\reference_data\\background_audio\\background_samples.csv'

In [21]:
from nade.io.detection_loader import find_detections_files, load_detections

detections_path = find_detections_files(PROJECT_ROOT / "tests/reference_data/raw_audio_songmeter",
                                        pattern = "*.csv")

df = load_detections(detections_path)

In [22]:
detections_path

[WindowsPath('c:/Users/ram212/PhD/Code/NADE/tests/reference_data/raw_audio_songmeter/site_1/BirdNET_RTable.csv'),
 WindowsPath('c:/Users/ram212/PhD/Code/NADE/tests/reference_data/raw_audio_songmeter/site_2/BirdNET_RTable.csv'),
 WindowsPath('c:/Users/ram212/PhD/Code/NADE/tests/reference_data/raw_audio_songmeter/site_3/BirdNET_RTable.csv')]

In [23]:
df

,filename,start_time,end_time,common_name,scientific_name,confidence,label
0,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,0.0,3.0,Fireworks,Fireworks,0.194369,NaN
1,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,3.0,6.0,Fireworks,Fireworks,0.192847,NaN
2,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,42.0,45.0,Yellow-tufted Pipit,Anthus crenatus,0.102397,NaN
3,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,51.0,54.0,Brant,Branta bernicla,0.100238,NaN
4,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,21.0,24.0,Common Crane,Grus grus,0.093708,NaN
...,...,...,...,...,...,...,...
1053,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,15.0,18.0,Hawaiian Petrel,Pterodroma sandwichensis,0.010742,NaN
1054,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,6.0,9.0,Horned Lark,Eremophila alpestris,0.010558,NaN
1055,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,51.0,54.0,Cassin's Sparrow,Peucaea cassinii,0.010544,NaN
1056,C:\Users\ram212\PhD\Code\NADE\tests\reference_...,0.0,3.0,Hawaiian Petrel,Pterodroma sandwichensis,0.010287,NaN


In [25]:
# Step 3: Optional, suggest foreground clips with high confidence for given species
# Normalise the foreground vocalisations to same amplitude (default -20dBFS)

from nade.sampling.foreground import normalise_foreground_vocalisations

normalise_foreground_vocalisations(
    PROJECT_ROOT / "tests/reference_data/foreground_audio",
    target_amplitude = -20
)

WindowsPath('C:/Users/ram212/PhD/Code/NADE/tests/reference_data/foreground_audio_normalised')

In [ ]:
# Step 4: combine foreground clips with background clips
#   Given a vocalisation amplitude range (default [-80dBFS to -20dBFS]), 
#   or a single ampltiude value,
#   or use the detections dataframe to load vocalisations above a certain confidence and measure the RMS 
#       in the vocalisation frequency range -- one average RMS per site
#   Output will be a bunch of combined audio files (in folder with site_name), whose filename contains the
#       background filename + foreground filename
#   These combined clips will have vocalisation amplitude as defined above 
#   There will also be a single csv output containing the filename, species_name, vocalisation_amplitude, datetime
#       (most of this info will already be in files_metadata)

In [ ]:
# Step 5: User runs their detection model on the combined audio files

In [ ]:
# Step 6: Generate the below output graphs:
#   Input: one covariate per site_name, combined audio detection scores
#   Output: scatter plot, x-axis is covariate, y-axis is detections score (binomial confidence interval)

#   Input: combined audio detection scores
#   Output: line plot, x-axis is datetime, y-axis is detections score (binomial confidence interval)

#   Input: combined audio detection scores
#   Output: line plot, x-axis is time of day, y-axis is detections score -- other option is a clock plot

#   Input: combined audio detection scores
#   Output: boxplot, x-axis is site name, y-axis is detection score

#   Input: combined audio detection scores
#   Output: line plot, x-axis is vocalisation amplitude, y-axis is detection score, one line per site

#   Input: one combined audio detection score for each species
#   Output: 